PHASE 1

In [1]:
import pandas as pd
import glob
import os
# combine all payment(BOLT) csv files into one file and creating a dataframe
files= glob.glob(r'C:\Users\olami\OneDrive\Desktop\LODZ RIDE METRICS\BOLT\bolt_payment\*.csv')
df_bolt_payments_raw = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
df_bolt_payments_raw.to_csv(r'C:\Users\olami\OneDrive\Desktop\LODZ RIDE METRICS\BOLT\bolt_payment\bolt_payment_stacked.csv', index=False)



In [2]:
# combine all ride(BOLT) csv files into one file and creating a dataframe 
path= r'C:\Users\olami\OneDrive\Desktop\LODZ RIDE METRICS\BOLT\Order history\order_history'
files = glob.glob(path + "/**/*.csv", recursive=True)
df_bolt_trips_raw = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
df_bolt_trips_raw.to_csv(r'C:\Users\olami\OneDrive\Desktop\LODZ RIDE METRICS\BOLT\Order history\order_history_stacked.csv', index=False)

In [3]:
# read the Uber trips csv file and creating a dataframe
path = r'C:\Users\olami\OneDrive\Desktop\LODZ RIDE METRICS\Uber\uber_trips.csv'
df_uber_trips_raw= pd.read_csv(path)

# read the Uber payments csv file and creating a dataframe
path = r"C:\Users\olami\OneDrive\Desktop\LODZ RIDE METRICS\Uber\uber_payments.csv"
df_uber_payments_raw= pd.read_csv(path)


In [21]:
import pyodbc
import pandas as pd

from sqlalchemy import create_engine

conn = pyodbc.connect(
    "Driver={ODBC Driver 18 for SQL Server};"
    "Server=LAMOSKI;"
    "Database=RideMetrics;"
    "Trusted_Connection=yes;"
    "Encrypt=no;"
)
engine = create_engine("mssql+pyodbc://LAMOSKI/RideMetrics?driver=ODBC+Driver+18+for+SQL+Server&trusted_connection=yes&encrypt=no")


cursor = conn.cursor()
cursor.execute("SELECT @@VERSION")
print(cursor.fetchone())

('Microsoft SQL Server 2022 (RTM) - 16.0.1000.6 (X64) \n\tOct  8 2022 05:58:25 \n\tCopyright (C) 2022 Microsoft Corporation\n\tDeveloper Edition (64-bit) on Windows 10 Home 10.0 <X64> (Build 26200: ) (Hypervisor)\n',)


REMOVAL OF PRIVATE SENSITIVE COLUMNS

In [22]:
# ingesting data into Bronze schema after prunning(private sensitive data) and selecting relevant columns.
#  Also saving the prunned dataframes as csv files in the RawFiles folder for future reference.

# Bolt payment data
df_bolt_payments_bronze = df_bolt_payments_raw[['Date', 'Payment method', 'Date of ride',  'Price (no VAT)', 'VAT', 'Price Total']]
df_bolt_payments_bronze.to_sql('Bolt_Payments', engine,schema='Bronze', if_exists= 'replace', index= False )
df_bolt_payments_bronze.to_csv(r'C:\Users\olami\OneDrive\Desktop\LODZ RIDE METRICS\RawFiles\Bolt_Payments.csv', index=False) 

# Bolt trips data
df_bolt_trips_bronze = df_bolt_trips_raw[['Time', 'Driver accepted', 'Ride finished', 'Pickup distance', 'Ride distance', 'Ride duration', 
                               'Order state', 'Reject reason', 'Pickup duration', 
                               'Estimated pickup time', 'Estimated distance', 'Estimated duration']]
df_bolt_trips_bronze.to_sql('Bolt_Trips', engine,schema= 'Bronze', if_exists= 'replace', index=False)
df_bolt_trips_bronze.to_csv(r'C:\Users\olami\OneDrive\Desktop\LODZ RIDE METRICS\RawFiles\Bolt_Trips.csv', index=False) 

# Uber trips data
df_uber_trips_bronze = df_uber_trips_raw[['global_product_name', 'status', 'request_timestamp_local',
                                'begintrip_timestamp_local', 'dropoff_timestamp_local',
                                'trip_distance_miles', 'trip_duration_seconds', 'base_fare_local',
                                'original_fare_local', 'cancellation_fee_local']]
df_uber_trips_bronze.to_sql('Uber_Trips', engine,schema='Bronze', if_exists= 'replace', index= False )
df_uber_trips_bronze.to_csv(r'C:\Users\olami\OneDrive\Desktop\LODZ RIDE METRICS\RawFiles\Uber_Trips.csv', index=False)

# Uber payments data
df_uber_payments_bronze = df_uber_payments_raw[['City Name', 'Trip UUID', 'Local Amount', 'Currency Code',
       'Classification', 'Category', 'Local Timestamp']]
df_uber_payments_bronze.to_sql('Uber_Payments', engine,schema='Bronze', if_exists= 'replace', index= False )
df_uber_payments_bronze.to_csv(r'C:\Users\olami\OneDrive\Desktop\LODZ RIDE METRICS\RawFiles\Uber_Payments.csv', index=False)


PHASE 2 (REMOVAL OF UNWANTED COLUMNS)

In [123]:
# reading the data from the Bronze schema to create dataframes for further processing in Silver and Gold layers.
df_bolt_trips_silver = pd.read_sql('select * from Bronze.Bolt_Trips',engine)
df_uber_trips_silver = pd.read_sql('select * from Bronze.Uber_Trips',engine)
df_uber_payments_silver = pd.read_sql('select * from Bronze.Uber_Payments',engine)

In [124]:
# selecting relevant columns for analysis
df_bolt_trips_silver = df_bolt_trips_silver[['Time', 'Driver accepted', 'Ride finished', 
       'Ride distance', 'Ride duration', 'Order state'
       ]]
df_uber_trips_silver = df_uber_trips_silver[[ 'status', 'request_timestamp_local',
       'begintrip_timestamp_local', 'dropoff_timestamp_local',
       'trip_distance_miles', 'trip_duration_seconds', 'base_fare_local',
       'original_fare_local', 'cancellation_fee_local']]
df_uber_payments_silver = df_uber_payments_silver[[ 'City Name', 'Trip UUID', 'Local Amount', 
       'Classification', 'Category', 'Local Timestamp']]



STANDADIZING DATETIME COLUMNS

In [125]:
# converting the Uber trips timestamp columns to datetime format for further analysis.
uber_trips_datetime_cols = ['request_timestamp_local', 'begintrip_timestamp_local',
       'dropoff_timestamp_local']
for i in uber_trips_datetime_cols:
       df_uber_trips_silver[i] = pd.to_datetime(df_uber_trips_silver[i], format='%Y-%m-%dT%H:%M:%S.%fZ',
        utc=True).dt.tz_localize(None)
       
#converting the Uber payments timestamp column to datetime format for further analysis.
df_uber_payments_silver['Local Timestamp'] = pd.to_datetime(df_uber_payments_silver['Local Timestamp'], format='mixed', errors='coerce')

# converting the Bolt trips timestamp columns to datetime format for further analysis.
bolt_trips_datetime_cols = ['Time', 'Driver accepted', 'Ride finished']
for i in bolt_trips_datetime_cols:
       df_bolt_trips_silver[i] = pd.to_datetime(df_bolt_trips_silver[i], format='%Y-%m-%d %H:%M:%S')




RENAMING COLUMNS FOR USER FRIENDLY

In [126]:
# Uber trips
df_uber_trips_silver.rename(columns={
    'status': 'trip_status',
    'request_timestamp_local': 'request_time',
    'begintrip_timestamp_local': 'start_time',
    'dropoff_timestamp_local': 'end_time',
    'trip_distance_miles': 'distance_miles',
    'trip_duration_seconds': 'duration_seconds',
    'base_fare_local': 'base_fare',
    'original_fare_local': 'gross_fare',
    'cancellation_fee_local': 'cancellation_fee'
}, inplace=True)

# Uber payments
df_uber_payments_silver.rename(columns={
    'City Name': 'city_name',
    'Trip UUID': 'trip_uuid',
    'Local Amount': 'amount',
    'Classification': 'classification',
    'Category': 'category',
    'Local Timestamp': 'local_timestamp'
}, inplace=True)

# Bolt trips
df_bolt_trips_silver.rename(columns={
    'Time': 'request_time',
    'Driver accepted': 'driver_accepted_time',
    'Ride finished': 'ride_finished_time',
    'Ride distance': 'ride_distance',
    'Ride duration': 'ride_duration',
    'Order state': 'order_state'
}, inplace=True)



DROPPING CANCELLED TRIPS


In [127]:
# Dropping cancelled trips from the Bolt trips dataframe to ensure that only completed trips are considered for analysis.
df_bolt_silver = df_bolt_trips_silver.dropna().copy()

PIVOTING UBER PAYMENT TABLE

In [128]:
# pivoting the Uber payments dataframe to have separate columns for each classification, 
# with the sum of amounts for each classification as the values.
df_uber_payments_silver_pivoted = df_uber_payments_silver.pivot_table(
    index=['local_timestamp', 'city_name'],
    columns='classification',
    values='amount',
    aggfunc='sum'
).fillna(0).reset_index()

# Fix column flattening
df_uber_payments_silver_pivoted.columns = [
    '_'.join(col).strip('_') if isinstance(col, tuple) else col
    for col in df_uber_payments_silver_pivoted.columns
]

# Define columns to exclude
exclude_cols = [
    'local_timestamp',
    'city_name',
    'transport.fare.cash.collected',
    'transport.misc.tip'
]

# Get all columns to sum
cols_to_sum = [col for col in df_uber_payments_silver_pivoted.columns 
               if col not in exclude_cols]

# Net earnings calculation: sum of all payment classifications except for cash collected.
df_uber_payments_silver_pivoted['net_earnings_no_tips'] = (
    df_uber_payments_silver_pivoted[cols_to_sum].sum(axis=1)
)
df_uber_payments_silver_pivoted['tips'] = (
    df_uber_payments_silver_pivoted['transport.misc.tip']
)

df_uber_payments_silver_pivoted['net_earnings_with_tips'] = df_uber_payments_silver_pivoted['net_earnings_no_tips']+ df_uber_payments_silver_pivoted['tips']

# keeping only the local_timestamp and net_earnings columns for further analysis. 
df_uber_payments_silver_pivoted = df_uber_payments_silver_pivoted[
    ['local_timestamp', 'net_earnings_no_tips', 'tips', 'net_earnings_with_tips']
]


PRUNING MY DATAFRAME TO LODZ ONLY 


In [129]:
#restricting the dataframes to only include records from September 23, 2024, 16:19:00 onwards for both Uber payments and trips data. this ensures data from lodz(including when lodz deduction started)
df_uber_payments_silver_pivoted = df_uber_payments_silver_pivoted[df_uber_payments_silver_pivoted['local_timestamp'] >= '2024-09-23 16:19:00']
df_uber_trips_silver= df_uber_trips_silver[df_uber_trips_silver['request_time'] >= '2024-09-23 16:19:00']

MERGE TABLES

In [130]:


# flooring the timestamp columns to the nearest minute for both Uber trips and payments dataframes to facilitate joining on a common time basis.
df_uber_trips_silver['request_time_rounded'] = df_uber_trips_silver['request_time'].dt.floor('min')
df_uber_payments_silver_pivoted['local_timestamp_rounded'] = df_uber_payments_silver_pivoted['local_timestamp'].dt.floor('min')

# joning the Uber trips and payments dataframes on the rounded timestamp columns to create a consolidated dataframe for analysis.
#automatically dropping the cancelled trips with no payments from the Uber trips dataframe during the join operation.
df_uber_silver = pd.merge(
    df_uber_trips_silver,
    df_uber_payments_silver_pivoted,
    left_on='request_time_rounded',
    right_on='local_timestamp_rounded',
    how='inner'
)





COMBINED TABLES(UBER AND BOLT)

In [131]:
# adding a new column to both Uber and Bolt dataframes to indicate the platform for further analysis.
df_uber_silver['platform'] = 'Uber'
df_bolt_silver['platform'] = 'Bolt' 

# converting uber distance duration column to metre to correspond with the Bolt distance duration column for further analysis.
df_uber_silver['distance_metres'] = (df_uber_silver['distance_miles'] * 1609.34).round(2)
df_uber_silver = df_uber_silver[['platform','request_time','end_time','distance_metres','duration_seconds',
                                'net_earnings_no_tips','tips','net_earnings_with_tips','gross_fare','trip_status']] 


In [132]:
# creating a combined dataframe for both Uber and Bolt trips data with common columns for further analysis.
uber_common = df_uber_silver[['platform',
    'request_time',
    'end_time',
    'distance_metres',
    'duration_seconds',
    'trip_status']].rename(columns={
        'distance_metres': 'distance',
        'duration_seconds': 'duration'
    })
bolt_common = df_bolt_silver[['platform',
    'request_time',
    'ride_finished_time',
    'ride_distance',
    'ride_duration',
    'order_state']].rename(columns={
        'ride_finished_time': 'end_time',
        'ride_distance': 'distance',
        'ride_duration': 'duration',
        'order_state': 'trip_status'
    })
df_combined_silver = pd.concat([uber_common, bolt_common], ignore_index=True)
df_combined_silver.to_sql('Combined_Silver', engine,schema='Silver', if_exists= 'replace', index= False )
df_uber_silver.to_sql('Uber', engine,schema='Silver', if_exists= 'replace', index= False )
df_bolt_silver.to_sql('Bolt', engine,schema='Silver', if_exists= 'replace', index= False )

101

GOLD LAYER

In [133]:
#creating a daily revenue summary for Uber trips data with relevant metrics for further analysis.
daily_revenue_gold = df_uber_silver.groupby(df_uber_silver['request_time'].dt.date).agg(
    total_trips = ('trip_status', 'count'),
    total_revenue= ('gross_fare', 'sum'),
    total_net_earnings_no_tips = ('net_earnings_no_tips', 'sum'),
    total_tips = ('tips', 'sum'),
    total_net_earnings_with_tips = ('net_earnings_with_tips', 'sum'),
    avg_net_earnings_no_tips = ('net_earnings_no_tips', 'mean'),
    distance_covered = ('distance_metres', 'sum'),
    avg_distance_per_trip = ('distance_metres', 'mean'),
    total_duration = ('duration_seconds', 'sum')). reset_index()

daily_revenue_gold.rename(columns={'request_time': 'date'}, inplace=True)

# Note: Uber deduction includes platform commission, VAT and service fees
daily_revenue_gold['uber_deduction'] = daily_revenue_gold['total_revenue'] - daily_revenue_gold['total_net_earnings_no_tips']
daily_revenue_gold['uber_deduction_%'] = (daily_revenue_gold['uber_deduction'] / daily_revenue_gold['total_revenue']) * 100

# calculating the earnings per kilometre for each day by dividing the total net earnings (excluding tips) by the total distance covered (in kilometres).
daily_revenue_gold['earnings_per_km'] = (daily_revenue_gold['total_net_earnings_no_tips'] / (daily_revenue_gold['distance_covered'] / 1000)).round(2)


daily_revenue_gold.to_sql('Daily_Revenue', engine,schema='Gold', if_exists= 'replace', index= False )

12

In [134]:
#hourly revenue summary for Uber trips data with relevant metrics for further analysis.
hourly_revenue_gold = df_uber_silver.groupby(df_uber_silver['request_time'].dt.hour).agg(
    total_trips = ('trip_status', 'count'),
    total_revenue= ('gross_fare', 'sum'),
    total_net_earnings_no_tips = ('net_earnings_no_tips', 'sum'),
    total_tips = ('tips', 'sum'),
    total_net_earnings_with_tips = ('net_earnings_with_tips', 'sum'),
    avg_net_earnings_no_tips = ('net_earnings_no_tips', 'mean'),
    distance_covered = ('distance_metres', 'sum'),
    avg_distance_per_trip = ('distance_metres', 'mean'),
    total_duration = ('duration_seconds', 'sum')). reset_index()

hourly_revenue_gold.rename(columns={'request_time': 'hour'}, inplace=True)

# Note: Uber deduction includes platform commission, VAT and service fees
hourly_revenue_gold['uber_deduction'] = hourly_revenue_gold['total_revenue'] - hourly_revenue_gold['total_net_earnings_no_tips']
hourly_revenue_gold['uber_deduction_%'] = (hourly_revenue_gold['uber_deduction'] / hourly_revenue_gold['total_revenue']) * 100

# calculating the earnings per kilometre for each hour by dividing the total net earnings (excluding tips) by the total distance covered (in kilometres).
hourly_revenue_gold['earnings_per_km'] = (hourly_revenue_gold['total_net_earnings_no_tips'] / (hourly_revenue_gold['distance_covered'] / 1000)).round(2)

hourly_revenue_gold.to_sql('Hourly_Revenue', engine,schema='Gold', if_exists= 'replace', index= False )

24

In [135]:
# monthly revenue summary for Uber trips data with relevant metrics for further analysis.
monthly_revenue_gold = df_uber_silver.groupby(df_uber_silver['request_time'].dt.month).agg(
    total_trips = ('trip_status', 'count'),
    total_revenue= ('gross_fare', 'sum'),
    total_net_earnings_no_tips = ('net_earnings_no_tips', 'sum'),
    total_tips = ('tips', 'sum'),
    total_net_earnings_with_tips = ('net_earnings_with_tips', 'sum'),
    avg_net_earnings_no_tips = ('net_earnings_no_tips', 'mean'),
    distance_covered = ('distance_metres', 'sum'),
    avg_distance_per_trip = ('distance_metres', 'mean'),
    total_duration = ('duration_seconds', 'sum')). reset_index()

monthly_revenue_gold.rename(columns={'request_time': 'month'}, inplace=True)

# Note: Uber deduction includes platform commission, VAT and service fees
monthly_revenue_gold['uber_deduction'] = monthly_revenue_gold['total_revenue'] - monthly_revenue_gold['total_net_earnings_no_tips']
monthly_revenue_gold['uber_deduction_%'] = (monthly_revenue_gold['uber_deduction'] / monthly_revenue_gold['total_revenue']) * 100

# calculating the earnings per kilometre for each month by dividing the total net earnings (excluding tips) by the total distance covered (in kilometres).
monthly_revenue_gold['earnings_per_km'] = (monthly_revenue_gold['total_net_earnings_no_tips'] / (monthly_revenue_gold['distance_covered'] / 1000)).round(2)

monthly_revenue_gold.to_sql('Monthly_Revenue', engine,schema='Gold', if_exists= 'replace', index= False )

12

In [136]:
# monthly revenue summary for Uber trips data with relevant metrics for further analysis.
yearly_monthly_revenue_gold = df_uber_silver.groupby(df_uber_silver['request_time'].dt.to_period('M')).agg(
    total_trips = ('trip_status', 'count'),
    total_revenue= ('gross_fare', 'sum'),
    total_net_earnings_no_tips = ('net_earnings_no_tips', 'sum'),
    total_tips = ('tips', 'sum'),
    total_net_earnings_with_tips = ('net_earnings_with_tips', 'sum'),
    avg_net_earnings_no_tips = ('net_earnings_no_tips', 'mean'),
    distance_covered = ('distance_metres', 'sum'),
    avg_distance_per_trip = ('distance_metres', 'mean'),
    total_duration = ('duration_seconds', 'sum')). reset_index()

yearly_monthly_revenue_gold.rename(columns={'request_time': 'month'}, inplace=True)
yearly_monthly_revenue_gold['month'] = yearly_monthly_revenue_gold['month'].astype(str)
# Note: Uber deduction includes platform commission, VAT and service fees
yearly_monthly_revenue_gold['uber_deduction'] = yearly_monthly_revenue_gold['total_revenue'] - yearly_monthly_revenue_gold['total_net_earnings_no_tips']
yearly_monthly_revenue_gold['uber_deduction_%'] = (yearly_monthly_revenue_gold['uber_deduction'] / yearly_monthly_revenue_gold['total_revenue']) * 100

# calculating the earnings per kilometre for each month by dividing the total net earnings (excluding tips) by the total distance covered (in kilometres).
yearly_monthly_revenue_gold['earnings_per_km'] = (yearly_monthly_revenue_gold['total_net_earnings_no_tips'] / (yearly_monthly_revenue_gold['distance_covered'] / 1000)).round(2)

yearly_monthly_revenue_gold.to_sql('Yearly_Monthly_Revenue', engine,schema='Gold', if_exists= 'replace', index= False )

17

In [137]:
# weekly_revenue summary for Uber trips data with relevant metrics for further analysis.
weekly_revenue_gold = df_uber_silver.groupby(pd.Grouper(key='request_time', freq='W')).agg(
    total_trips = ('trip_status', 'count'),
    total_revenue= ('gross_fare', 'sum'),
    total_net_earnings_no_tips = ('net_earnings_no_tips', 'sum'),
    total_tips = ('tips', 'sum'),
    total_net_earnings_with_tips = ('net_earnings_with_tips', 'sum'),
    avg_net_earnings_no_tips = ('net_earnings_no_tips', 'mean'),
    distance_covered = ('distance_metres', 'sum'),
    avg_distance_per_trip = ('distance_metres', 'mean'),
    total_duration = ('duration_seconds', 'sum')). reset_index()

weekly_revenue_gold.rename(columns={'request_time': 'week'}, inplace=True)

# Note: Uber deduction includes platform commission, VAT and service fees
weekly_revenue_gold['uber_deduction'] = weekly_revenue_gold['total_revenue'] - weekly_revenue_gold['total_net_earnings_no_tips']
weekly_revenue_gold['uber_deduction_%'] = (weekly_revenue_gold['uber_deduction'] / weekly_revenue_gold['total_revenue']) * 100

# calculating the earnings per kilometre for each week by dividing the total net earnings (excluding tips) by the total distance covered (in kilometres).
weekly_revenue_gold['earnings_per_km'] = (weekly_revenue_gold['total_net_earnings_no_tips'] / (weekly_revenue_gold['distance_covered'] / 1000)).round(2)

weekly_revenue_gold.to_sql('Weekly_Revenue', engine,schema='Gold', if_exists= 'replace', index= False )

69

In [138]:
#seasonal demand summary for both Uber and Bolt trips data with relevant metrics for further analysis.
df_combined_silver['month']= df_combined_silver['request_time'].dt.month
df_combined_silver['season']= df_combined_silver['month'].map({
    1: 'Winter',2: 'Winter', 3:'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8:'Summer', 9:'Autumn', 10:'Autumn',11: 'Autumn', 12: 'Winter' })

seasonal_demand_gold =  df_combined_silver.groupby(['season']).agg(
    total_trips = ('request_time', 'count'),
    avg_distance = ('distance', 'mean'),
     avg_duration = ('duration', 'mean')
).reset_index()

seasonal_demand_gold.to_sql('Seasonal_Demand', engine,schema='Gold', if_exists= 'replace', index= False )

4

In [139]:
#hourly demand summary for both Uber and Bolt trips data with relevant metrics for further analysis.
df_combined_silver['hour'] = df_combined_silver['request_time'].dt.hour

hourly_demand_gold =  df_combined_silver.groupby(['hour']).agg(
    total_trips = ('request_time', 'count'),
    avg_distance = ('distance', 'mean'),
     avg_duration = ('duration', 'mean')
).reset_index()

hourly_demand_gold.to_sql('Hourly_Demand', engine,schema='Gold', if_exists= 'replace', index= False )

24

In [140]:
#daily demand summary for both Uber and Bolt trips data with relevant metrics for further analysis.
df_combined_silver['day_of_week'] = df_combined_silver['request_time'].dt.day_name()

daily_demand_gold =  df_combined_silver.groupby(['day_of_week']).agg(
    total_trips = ('request_time', 'count'),
    avg_distance = ('distance', 'mean'),
     avg_duration = ('duration', 'mean')
).reset_index()

daily_demand_gold.to_sql('Daily_Demand', engine,schema='Gold', if_exists= 'replace', index= False )

7

EXPLORATORY DATA ANALYSIS(EDA)

In [141]:
# checking the shape,null values and duplicates of all gold tables to ensure data integrity and completeness.
gold_tables = {
    'daily_revenue' : daily_revenue_gold,
    'hourly_revenue' : hourly_revenue_gold,
    'weekly_revenue': weekly_revenue_gold,
    'monthly_revenue' : monthly_revenue_gold,
    'yearly_monthly_revenue': yearly_monthly_revenue_gold,
    'daily_demand' :daily_demand_gold,
    'hourly_demand' : hourly_demand_gold,
    'seasonal_demand' : seasonal_demand_gold

}
print(f"{'name':<25}{'shape':<25}{'null':<10}{'duplicates':<10}")
print('-'*70)
for name, df in gold_tables.items():
    nulls = df.isnull().sum().sum()
    duplicates = df.duplicated().sum()
    print(f"{(name):<25}{str(df.shape):<25}{nulls:<10}{duplicates:<10}")

name                     shape                    null      duplicates
----------------------------------------------------------------------
daily_revenue            (334, 13)                0         0         
hourly_revenue           (24, 13)                 0         0         
weekly_revenue           (69, 13)                 16        0         
monthly_revenue          (12, 13)                 0         0         
yearly_monthly_revenue   (17, 13)                 0         0         
daily_demand             (7, 4)                   0         0         
hourly_demand            (24, 4)                  0         0         
seasonal_demand          (4, 4)                   0         0         


In [115]:
#dropping the weeks with no trips from the weekly revenue summary to ensure that only weeks with completed trips are considered for analysis.
weekly_revenue_gold = weekly_revenue_gold[weekly_revenue_gold['total_trips']>0].reset_index(drop=True)
print(weekly_revenue_gold.isnull().sum().sum())

0


In [122]:
weekly_revenue_gold.shape

(65, 13)

In [116]:
#checking the data types of all gold tables to ensure consistency and correctness of data types for further analysis.
for name, df in gold_tables.items():
    print(f"---{name} dtypes---")
    print(df.dtypes)
    print("\n")

---daily_revenue dtypes---
date                             object
total_trips                       int64
total_revenue                   float64
total_net_earnings_no_tips      float64
total_tips                      float64
total_net_earnings_with_tips    float64
avg_net_earnings_no_tips        float64
distance_covered                float64
avg_distance_per_trip           float64
total_duration                  float64
uber_deduction                  float64
uber_deduction_%                float64
earnings_per_km                 float64
dtype: object


---hourly_revenue dtypes---
hour                              int32
total_trips                       int64
total_revenue                   float64
total_net_earnings_no_tips      float64
total_tips                      float64
total_net_earnings_with_tips    float64
avg_net_earnings_no_tips        float64
distance_covered                float64
avg_distance_per_trip           float64
total_duration                  float64
uber_dedu

In [142]:
for name, df in gold_tables.items():
    print(f"---top 3 and bottom 3 of {name}---")
    print("\nTop 3:")
    print(df.head(3))
    print("\nBottom 3:")
    print(df.tail(3))
    print("\n")

---top 3 and bottom 3 of daily_revenue---

Top 3:
         date  total_trips  total_revenue  total_net_earnings_no_tips  \
0  2024-09-23            5          86.25                       65.11   
1  2024-09-25            8         236.69                      167.34   
2  2024-09-26           17         531.98                      377.66   

   total_tips  total_net_earnings_with_tips  avg_net_earnings_no_tips  \
0         0.0                         65.11                 13.022000   
1         0.0                        167.34                 20.917500   
2         0.0                        377.66                 22.215294   

   distance_covered  avg_distance_per_trip  total_duration  uber_deduction  \
0          27734.91            5546.982000          3043.0           21.14   
1          56786.87            7098.358750          8825.0           69.35   
2         130961.27            7703.604118         18740.0          154.32   

   uber_deduction_%  earnings_per_km  
0         24

In [ ]:
#changing the data type of the date column in the daily revenue summary to datetime format for further analysis.
daily_revenue_gold['date']= pd.to_datetime(daily_revenue_gold['date'])
print(daily_revenue_gold.dtypes['date'])

datetime64[s]


In [ ]:
daily_revenue_gold
weekly_revenue_gold
hourly_revenue_gold
monthly_revenue_gold
yearly_monthly_revenue_gold
daily_demand_gold
hourly_demand_gold
seasonal_demand_gold